# Kémzy àvátâr — PersonaLive CUDA proof v3

Kaggle-native environment setup. Uses Kaggle's preinstalled Python 3.12 + CUDA PyTorch stack; no conda, no venv, no uv-managed Python, and no duplicate CUDA wheels.

In [ ]:
import os, sys, subprocess, pathlib, shutil
ROOT='/kaggle/working'
KEMZY=f'{ROOT}/Kemzy-LiveAvatar'
PL=f'{ROOT}/PersonaLive'
PL_COMMIT='abdd112e01dcf7d89122c2e5efa29fcff0669740'
def run(cmd, **kw):
    print('+', ' '.join(map(str, cmd)))
    return subprocess.run(cmd, check=True, **kw)
for path in (KEMZY, PL):
    if os.path.isdir(path): shutil.rmtree(path)
run(['git','clone','--depth','1','https://github.com/eneokonaniebiet/Kemzy-LiveAvatar.git',KEMZY])
run(['git','clone','https://github.com/GVCLab/PersonaLive.git',PL])
run(['git','-C',PL,'fetch','--depth','1','origin',PL_COMMIT])
run(['git','-C',PL,'checkout','--detach',PL_COMMIT])
print('Kaggle Python:', sys.version)
print('PersonaLive commit:', subprocess.check_output(['git','-C',PL,'rev-parse','HEAD'],text=True).strip())
run(['nvidia-smi'])


In [ ]:
import importlib.util, subprocess, sys, os
import torch
print('Kaggle torch:', torch.__version__)
print('Torch CUDA:', torch.version.cuda, 'available=', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'Kaggle CUDA GPU is not available'
req=f'{KEMZY}/tools/personalive_kaggle_requirements.txt'
assert os.path.isfile(req), req
run([sys.executable,'-m','pip','install','--no-cache-dir','-r',req])
try:
    import mediapipe as mp
    print('MediaPipe:', getattr(mp,'__version__','unknown'), '(Kaggle-provided)')
except Exception:
    run([sys.executable,'-m','pip','install','--no-cache-dir','mediapipe==0.10.35'])
    import mediapipe as mp
    print('MediaPipe:', getattr(mp,'__version__','unknown'))


In [ ]:
mods=['accelerate','av','decord','diffusers','einops','fastapi','huggingface_hub','mediapipe','markdown2','numpy','omegaconf','cv2','PIL','polygraphy','pydantic','safetensors','skimage','starlette','tqdm','transformers','peft']
for name in mods:
    m=__import__(name)
    print(f'{name}: OK', getattr(m,'__version__',''))
os.chdir(PL)
sys.path.insert(0,PL)
spec=importlib.util.spec_from_file_location('personalive_offline',f'{PL}/inference_offline.py')
mod=importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
print('PersonaLive inference_offline import: PASS')
print('Kaggle-native environment setup: PASS')


## Expected result

The notebook should reach `Kaggle-native environment setup: PASS` without creating a virtual environment and without reinstalling Torch, torchvision, xformers, or CUDA component wheels. Model-weight/render tests can then run in a separate cell using the already-prepared environment.